# Clinical Dashboard — Symptom-Aware Temporal Exploration

This notebook replaces the statistical lens (PCA, variance) with a **clinical lens** (DSM-5 symptoms, 
semantic polarization, text attribution). Every axis, every color, every drill-down answers a 
**clinical question**, not a statistical one.

### Design principles

1. **Symptom axes, not PCA axes** — trajectories live in DSM-5 anchor space (anhedonia, hopelessness, sleep...)
2. **Interactive HNSW** — select level, click region → see representative posts + symptom distribution
3. **Text attribution** — when something changes (changepoint, drift), show *what* changed in language
4. **Polarization tracking** — semantic area contraction = obsessive focus = clinical risk
5. **Connected panels** — user deep-dive links all views into a unified clinical profile

### CVX functions used

| Panel | CVX Functions | Clinical Question |
|-------|--------------|-------------------|
| Symptom Trajectory | `project_to_anchors`, `anchor_summary` | Where is this user drifting in symptom space? |
| HNSW Explorer | `regions`, `region_members` | What semantic clusters exist and what's their clinical content? |
| Clinical Timeline | `velocity`, `detect_changepoints`, `drift` | When did language change and what changed? |
| Polarization | `trajectory` → dispersion | Is this user's world shrinking? |
| Cohort Divergence | `project_to_anchors` | When do depressed users diverge from controls? |
| Clinical Profile | All of the above | Full clinical picture of one user |

In [1]:
import chronos_vector as cvx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm
import json, time, os, warnings
import importlib, chronos_vector                                                                                                                                     

importlib.reload(chronos_vector)
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
EMB_DIR = f'{DATA_DIR}/embeddings'
CACHE_DIR = f'{DATA_DIR}/cache'

C_DEP = '#e74c3c'
C_CTL = '#3498db'
C_ACCENT = '#2ecc71'
C_WARN = '#f39c12'
C_SYMPTOM = '#9b59b6'

SYMPTOM_LABELS = [
    'Depressed\nMood', 'Anhedonia', 'Sleep', 'Fatigue',
    'Worthlessness', 'Concentration', 'Suicidal\nIdeation',
    'Appetite', 'Psychomotor', 'Healthy'
]
SYMPTOM_KEYS = [
    'depressed_mood', 'anhedonia', 'sleep_disturbance', 'fatigue',
    'worthlessness', 'concentration', 'suicidal_ideation',
    'appetite', 'psychomotor'
]

---
## 1. Data Loading

Reuse the cached CVX index from B2 (1.3M posts, 2285 users, D=768 MentalRoBERTa)
and the pre-computed DSM-5 anchor vectors.

In [2]:
# Load metadata only (skip embedding columns to save ~8GB RAM)
import re

df_full = pd.read_parquet(f'{EMB_DIR}/erisk_mental_embeddings.parquet')
emb_cols = [c for c in df_full.columns if c.startswith('emb_')]
D = len(emb_cols)
meta_cols = [c for c in df_full.columns if not c.startswith('emb_')]
df = df_full[meta_cols].copy()
del df_full  # free ~8GB immediately

user_to_id = {u: i for i, u in enumerate(sorted(df['user_id'].unique()))}
id_to_user = {v: k for k, v in user_to_id.items()}
df['entity_id'] = df['user_id'].map(user_to_id).astype(np.uint64)
user_label = dict(zip(df['user_id'], df['label']))

print(f'{len(df):,} posts, {df["user_id"].nunique()} users, D={D}')
print(f'Depression: {sum(1 for v in user_label.values() if v=="depression")} users')
print(f'Control: {sum(1 for v in user_label.values() if v=="control")} users')

# Load raw texts for attribution
# Parquet user_ids have edition prefixes (e2022_subject0, e2017train_train_subject6146)
# unified.jsonl has original IDs (subject0, train_subject6146)
# Build a mapping: parquet_uid → jsonl_uid
def strip_erisk_prefix(uid):
    """e2022_subject0 → subject0, e2017train_train_subject6146 → train_subject6146"""
    m = re.match(r'^e\d{4}(?:train|test)?_(.+)$', uid)
    return m.group(1) if m else uid

# Build texts dict keyed by PARQUET user_ids for direct lookup
raw_texts = {}
with open(f'{DATA_DIR}/eRisk/unified.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        raw_texts[(rec['user_id'], rec['timestamp'])] = rec['text']

# Also try 2018 unified if it exists
jsonl_2018 = f'{DATA_DIR}/eRisk/2018_unified.jsonl'
if os.path.exists(jsonl_2018):
    with open(jsonl_2018) as f:
        for line in f:
            rec = json.loads(line)
            raw_texts[(rec['user_id'], rec['timestamp'])] = rec['text']

# Build reverse mapping: for each parquet uid, find the jsonl uid
uid_to_jsonl = {uid: strip_erisk_prefix(uid) for uid in df['user_id'].unique()}

# Rekey texts using parquet UIDs
texts = {}
for (jsonl_uid, ts_str), txt in raw_texts.items():
    texts[(jsonl_uid, ts_str)] = txt

print(f'Loaded {len(texts):,} text snippets')
print(f'Sample mapping: {list(uid_to_jsonl.items())[:3]}')

1,359,188 posts, 2285 users, D=768
Depression: 233 users
Control: 2052 users
Loaded 1,899,243 text snippets
Sample mapping: [('e2022_subject0', 'subject0'), ('e2022_subject1000', 'subject1000'), ('e2022_subject1004', 'subject1004')]


In [3]:
# Load or rebuild CVX index
INDEX_PATH = f'{CACHE_DIR}/erisk_b2_full_index.cvx'

# Delete stale cache — timestamps were stored as microseconds, need seconds
if os.path.exists(INDEX_PATH):
    os.remove(INDEX_PATH)
    print('Removed stale index cache (timestamp fix)')

print('Rebuilding index with correct unix-second timestamps...')
df_emb = pd.read_parquet(f'{EMB_DIR}/erisk_mental_embeddings.parquet')
emb_cols_list = [c for c in df_emb.columns if c.startswith('emb_')]

index = cvx.TemporalIndex(m=16, ef_construction=50, ef_search=50)
entity_ids = df_emb['user_id'].map(user_to_id).values.astype(np.uint64)
# CRITICAL: datetime64[us] → int64 gives microseconds. Divide by 10^6 for seconds.
timestamps = (df_emb['timestamp'].astype('int64') // 10**6).values.astype(np.int64)
vectors = df_emb[emb_cols_list].values.astype(np.float32)

print(f'Timestamp range: {timestamps.min()} to {timestamps.max()}')
print(f'  = {pd.Timestamp(timestamps.min(), unit="s")} to {pd.Timestamp(timestamps.max(), unit="s")}')

t0 = time.perf_counter()
index.bulk_insert(entity_ids, timestamps, vectors, ef_construction=25)
elapsed = time.perf_counter() - t0
print(f'Built in {elapsed:.0f}s ({len(index):,} pts, {len(index)/elapsed:.0f} pts/sec)')

os.makedirs(os.path.dirname(INDEX_PATH), exist_ok=True)
index.save(INDEX_PATH)
print(f'Cached to {INDEX_PATH}')
del df_emb, vectors

# Load DSM-5 anchor vectors
ANCHOR_CACHE = f'{CACHE_DIR}/dsm5_anchors.npz'
data = np.load(ANCHOR_CACHE, allow_pickle=True)
anchor_vectors = data['anchor_vectors'].item()
healthy_vector = data['healthy_vector']

# Global centroid for centering (see RFC-012 Part B)
CENTROID_CACHE = f'{CACHE_DIR}/erisk_mental_centroid.npy'
if os.path.exists(CENTROID_CACHE):
    global_centroid = np.load(CENTROID_CACHE)
    print(f'Loaded global centroid (norm={np.linalg.norm(global_centroid):.4f})')
else:
    print('Computing global embedding centroid...')
    df_emb = pd.read_parquet(f'{EMB_DIR}/erisk_mental_embeddings.parquet')
    emb_cols_list = [c for c in df_emb.columns if c.startswith('emb_')]
    global_centroid = df_emb[emb_cols_list].values.astype(np.float32).mean(axis=0)
    np.save(CENTROID_CACHE, global_centroid)
    print(f'Centroid cached (norm={np.linalg.norm(global_centroid):.4f})')
    del df_emb

# Centered + normalized anchor matrix
def _cn(vec):
    v = np.asarray(vec, dtype=np.float32)
    v = v / np.linalg.norm(v) - global_centroid
    return v / np.linalg.norm(v)

anchor_names_ordered = SYMPTOM_KEYS + ['healthy']
raw_anchors = {k: anchor_vectors[k] for k in SYMPTOM_KEYS}
raw_anchors['healthy'] = healthy_vector
anchor_matrix_centered = np.array([_cn(raw_anchors[k]) for k in anchor_names_ordered], dtype=np.float32)

print(f'{len(SYMPTOM_KEYS)} symptom anchors + 1 healthy baseline (centered)')

# Centered projection functions
def project_centered(traj):
    """Project trajectory to centered anchor space."""
    results = []
    for ts, vec in traj:
        v = np.asarray(vec, dtype=np.float32) - global_centroid
        v = v / (np.linalg.norm(v) + 1e-8)
        sims = v @ anchor_matrix_centered.T
        dists = 1.0 - sims
        results.append((ts, dists.tolist()))
    return results

def anchor_summary_centered(projected):
    """Mean/min/trend per anchor from centered projection."""
    dists = np.array([d for _, d in projected])
    n = len(dists)
    means = dists.mean(axis=0).tolist()
    mins = dists.min(axis=0).tolist()
    x = np.arange(n, dtype=np.float32)
    x_mean = x.mean()
    trends = []
    for j in range(dists.shape[1]):
        y = dists[:, j]
        slope = np.sum((x - x_mean) * (y - y.mean())) / (np.sum((x - x_mean)**2) + 1e-8)
        trends.append(float(slope))
    return {'mean': means, 'min': mins, 'trend': trends}

# Quick timestamp test
test_traj = index.trajectory(entity_id=list(user_to_id.values())[0])
test_ts = test_traj[0][0]
print(f'Index timestamp test: {test_ts} = {pd.Timestamp(test_ts, unit="s")}')

Removed stale index cache (timestamp fix)
Rebuilding index with correct unix-second timestamps...
Timestamp range: -9223372036855 to 1640038650
  = -290308-12-21 19:59:05 to 2021-12-20 22:17:30
Built in 1574s (1,359,188 pts, 864 pts/sec)
Cached to ../data/cache/erisk_b2_full_index.cvx
Loaded global centroid (norm=0.9646)
9 symptom anchors + 1 healthy baseline (centered)
Index timestamp test: 1425349631 = 2015-03-03 02:27:11


In [4]:
# Helper: get text for a user at a given unix timestamp
def get_text(uid, unix_ts, max_len=200):
    """Look up post text. Handles parquet→jsonl uid mapping."""
    try:
        ts_str = pd.Timestamp(unix_ts, unit='s').strftime('%Y-%m-%d %H:%M:%S')
    except (ValueError, NotImplementedError, OverflowError):
        return ''
    txt = texts.get((uid, ts_str), '')
    if not txt:
        txt = texts.get((uid_to_jsonl.get(uid, uid), ts_str), '')
    return txt[:max_len] if txt else ''

# Helper: project a user's trajectory to centered anchor space
def user_anchor_trajectory(uid, cutoff_frac=1.0):
    """Returns (traj, timestamps, distances) in centered DSM-5 space."""
    eid = user_to_id[uid]
    traj = index.trajectory(entity_id=eid)
    if len(traj) < 5:
        return None, None, None
    n_use = max(5, int(len(traj) * cutoff_frac))
    traj = traj[:n_use]
    
    projected = project_centered(traj)
    timestamps = np.array([ts for ts, _ in traj])
    distances = np.array([dists for _, dists in projected])
    return traj, timestamps, distances

# Helper: get posts with text
def user_posts_with_text(uid, traj=None):
    if traj is None:
        traj = index.trajectory(entity_id=user_to_id[uid])
    rows = []
    for ts, vec in traj:
        txt = get_text(uid, ts, max_len=300)
        rows.append({
            'timestamp': pd.Timestamp(ts, unit='s'),
            'unix_ts': ts,
            'text': txt,
            'text_short': txt[:120] + '...' if len(txt) > 120 else txt,
        })
    return pd.DataFrame(rows)

# Quick test
test_uid = list(user_to_id.keys())[0]
test_traj = index.trajectory(entity_id=user_to_id[test_uid])
test_text = get_text(test_uid, test_traj[0][0], 100)
test_proj = project_centered(test_traj[:1])
test_dist = test_proj[0][1][0]
print(f'Text test: "{test_text[:60]}..."' if test_text else f'Text FAILED for {test_uid}')
print(f'Centered distance to depressed_mood: {test_dist:.4f} (should be 0.3-0.9, not 0.04-0.06)')
print('Helpers ready')

Text test: "What're you referring to exactly? I feel like the obvious an..."
Centered distance to depressed_mood: 0.8392 (should be 0.3-0.9, not 0.04-0.06)
Helpers ready


---
## 2. Symptom Radar — Population-Level Clinical Profile

**Clinical question:** Which DSM-5 symptoms are users closest to, and how does this differ between depression and control groups?

Instead of PCA-reduced scatter plots, we project every user's mean embedding into DSM-5 anchor space using `cvx.project_to_anchors()`. The radar chart shows the **average symptom proximity profile** — each axis is a clinically defined symptom, not a statistical component.

In [5]:
# Compute centered anchor summary for all users
print('Computing centered anchor projections for all users...')

user_summaries = {}
user_post_counts = {}

for uid in tqdm(df['user_id'].unique(), desc='Anchor projection'):
    eid = user_to_id[uid]
    traj = index.trajectory(entity_id=eid)
    if len(traj) < 10:
        continue
    user_post_counts[uid] = len(traj)
    projected = project_centered(traj)
    user_summaries[uid] = anchor_summary_centered(projected)

print(f'Projected {len(user_summaries)} users (centered)')

def get_traj(uid):
    """Get trajectory from index on-demand."""
    return index.trajectory(entity_id=user_to_id[uid])

Computing centered anchor projections for all users...


Anchor projection:   0%|          | 0/2285 [00:00<?, ?it/s]

Projected 2263 users (centered)


In [6]:
# Radar chart: mean symptom proximity — Depression vs Control
# We use (1 - cosine_distance) as "proximity" so higher = closer to symptom

dep_profiles = []
ctl_profiles = []
for uid, summary in user_summaries.items():
    proximity = [1.0 - d for d in summary['mean']]  # convert distance → proximity
    if user_label[uid] == 'depression':
        dep_profiles.append(proximity)
    else:
        ctl_profiles.append(proximity)

dep_mean = np.mean(dep_profiles, axis=0)
ctl_mean = np.mean(ctl_profiles, axis=0)
dep_std = np.std(dep_profiles, axis=0)
ctl_std = np.std(ctl_profiles, axis=0)

# Radar labels (close the polygon)
radar_labels = SYMPTOM_LABELS + [SYMPTOM_LABELS[0]]
dep_r = list(dep_mean) + [dep_mean[0]]
ctl_r = list(ctl_mean) + [ctl_mean[0]]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=dep_r, theta=radar_labels, fill='toself',
    name=f'Depression (n={len(dep_profiles)})',
    line=dict(color=C_DEP, width=3),
    fillcolor='rgba(231, 76, 60, 0.15)',
))
fig.add_trace(go.Scatterpolar(
    r=ctl_r, theta=radar_labels, fill='toself',
    name=f'Control (n={len(ctl_profiles)})',
    line=dict(color=C_CTL, width=3),
    fillcolor='rgba(52, 152, 219, 0.15)',
))

fig.update_layout(
    title='DSM-5 Symptom Proximity Profile — Population Average',
    polar=dict(
        radialaxis=dict(visible=True, range=[0, max(max(dep_r), max(ctl_r)) * 1.1]),
        bgcolor='rgba(0,0,0,0)',
    ),
    width=750, height=600,
    template='plotly_dark',
    legend=dict(x=0.02, y=0.98),
    annotations=[dict(
        text='Higher = closer to symptom concept<br>Axes = DSM-5 depression criteria',
        x=0.5, y=-0.05, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='gray'),
    )],
)
fig.show()

In [7]:
# Symptom trend radar: who is APPROACHING symptoms over time?
# Negative trend = approaching (distance decreasing), so we negate for "drift toward"
dep_trends = []
ctl_trends = []
for uid, summary in user_summaries.items():
    trend_toward = [-t for t in summary['trend']]  # negate: positive = approaching
    if user_label[uid] == 'depression':
        dep_trends.append(trend_toward)
    else:
        ctl_trends.append(trend_toward)

dep_trend_mean = np.mean(dep_trends, axis=0)
ctl_trend_mean = np.mean(ctl_trends, axis=0)
dep_tr = list(dep_trend_mean) + [dep_trend_mean[0]]
ctl_tr = list(ctl_trend_mean) + [ctl_trend_mean[0]]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=dep_tr, theta=radar_labels, fill='toself',
    name='Depression', line=dict(color=C_DEP, width=3),
    fillcolor='rgba(231, 76, 60, 0.15)',
))
fig.add_trace(go.Scatterpolar(
    r=ctl_tr, theta=radar_labels, fill='toself',
    name='Control', line=dict(color=C_CTL, width=3),
    fillcolor='rgba(52, 152, 219, 0.15)',
))

fig.update_layout(
    title='Symptom Drift Direction — Who Is Approaching Symptoms Over Time?',
    polar=dict(
        radialaxis=dict(visible=True),
        bgcolor='rgba(0,0,0,0)',
    ),
    width=750, height=600,
    template='plotly_dark',
    annotations=[dict(
        text='Higher = drifting TOWARD that symptom over time<br>Negative slope in anchor distance = approaching',
        x=0.5, y=-0.05, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='gray'),
    )],
)
fig.show()

---
## 3. Symptom Drift Over Time — Small Multiples

**Clinical question:** How does proximity to each symptom evolve over the posting history?

The radar showed that static proximity is similar between groups. The drift radar showed
depression users are approaching symptoms faster. This panel shows **how** — one subplot per
DSM-5 symptom, with smoothed proximity curves for depression vs control cohorts over
normalized time. This bridges the static radar with the per-user timeline.

In [8]:
# Compute per-symptom proximity over normalized time — CENTERED
n_time_bins = 20
dep_users_list = [uid for uid in user_post_counts if user_label[uid] == 'depression'
                  and user_post_counts[uid] >= 20]
ctl_users_list = [uid for uid in user_post_counts if user_label[uid] == 'control'
                  and user_post_counts[uid] >= 20]

np.random.seed(42)
if len(ctl_users_list) > 300:
    ctl_users_list = list(np.random.choice(ctl_users_list, 300, replace=False))

all_cohort = dep_users_list + ctl_users_list
print(f'Small multiples: {len(dep_users_list)} dep + {len(ctl_users_list)} ctl')

dep_curves = np.zeros((n_time_bins, 10))
ctl_curves = np.zeros((n_time_bins, 10))
dep_n = np.zeros(n_time_bins)
ctl_n = np.zeros(n_time_bins)

for uid in tqdm(all_cohort, desc='Cohort temporal projection'):
    traj = get_traj(uid)
    projected = project_centered(traj)
    timestamps = np.array([ts for ts, _ in traj])
    proximities = 1.0 - np.array([d for _, d in projected])
    
    t_min, t_max = timestamps[0], timestamps[-1]
    if t_max == t_min:
        continue
    t_rel = (timestamps - t_min) / (t_max - t_min)
    bins = np.clip((t_rel * n_time_bins).astype(int), 0, n_time_bins - 1)
    
    is_dep = user_label[uid] == 'depression'
    for b in range(n_time_bins):
        mask = bins == b
        if mask.sum() > 0:
            if is_dep:
                dep_curves[b] += proximities[mask].mean(axis=0)
                dep_n[b] += 1
            else:
                ctl_curves[b] += proximities[mask].mean(axis=0)
                ctl_n[b] += 1

dep_n[dep_n == 0] = 1
ctl_n[ctl_n == 0] = 1
dep_curves /= dep_n[:, None]
ctl_curves /= ctl_n[:, None]
time_pct = np.linspace(5, 95, n_time_bins)

Small multiples: 213 dep + 300 ctl


Cohort temporal projection:   0%|          | 0/513 [00:00<?, ?it/s]

In [9]:
# Small multiples: one panel per DSM-5 symptom showing temporal evolution
fig = make_subplots(
    rows=3, cols=3, shared_xaxes=True, shared_yaxes=True,
    subplot_titles=[s.replace('\n', ' ') for s in SYMPTOM_LABELS[:9]],
    vertical_spacing=0.08, horizontal_spacing=0.05,
)

for idx in range(9):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    fig.add_trace(go.Scatter(
        x=time_pct, y=dep_curves[:, idx],
        mode='lines', line=dict(color=C_DEP, width=2.5),
        name='Depression' if idx == 0 else None,
        showlegend=(idx == 0),
        hovertemplate=f'{SYMPTOM_LABELS[idx].replace(chr(10), " ")}<br>'
                      'Time: %{x:.0f}%<br>Proximity: %{y:.4f}<extra>Depression</extra>',
    ), row=row, col=col)
    
    fig.add_trace(go.Scatter(
        x=time_pct, y=ctl_curves[:, idx],
        mode='lines', line=dict(color=C_CTL, width=2.5),
        name='Control' if idx == 0 else None,
        showlegend=(idx == 0),
        hovertemplate=f'{SYMPTOM_LABELS[idx].replace(chr(10), " ")}<br>'
                      'Time: %{x:.0f}%<br>Proximity: %{y:.4f}<extra>Control</extra>',
    ), row=row, col=col)
    
    # Shade the gap where depression > control
    fig.add_trace(go.Scatter(
        x=list(time_pct) + list(time_pct[::-1]),
        y=list(dep_curves[:, idx]) + list(ctl_curves[::-1, idx]),
        fill='toself', fillcolor='rgba(231, 76, 60, 0.1)',
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=row, col=col)

fig.update_layout(
    title='Symptom Proximity Over Normalized Time — Depression (red) vs Control (blue)',
    height=750, width=1000, template='plotly_dark',
    legend=dict(x=0.85, y=1.0),
)

for i in range(1, 4):
    fig.update_xaxes(title_text='% of post history', row=3, col=i)
for i in range(1, 4):
    fig.update_yaxes(title_text='Proximity', row=i, col=1)

fig.show()

# Select sample users for later panels
dep_users = [uid for uid in user_post_counts if user_label[uid] == 'depression'
             and 50 <= user_post_counts[uid] <= 500]
ctl_users = [uid for uid in user_post_counts if user_label[uid] == 'control'
             and 50 <= user_post_counts[uid] <= 500]
np.random.seed(42)
sample_dep = np.random.choice(dep_users, size=min(5, len(dep_users)), replace=False)
sample_ctl = np.random.choice(ctl_users, size=min(5, len(ctl_users)), replace=False)
print(f'\nSample users for later panels: {len(sample_dep)} dep + {len(sample_ctl)} ctl')


Sample users for later panels: 5 dep + 5 ctl


---
## 4. Interactive HNSW Explorer

**Clinical question:** What semantic clusters exist in the data, and what is their clinical content?

The HNSW graph creates natural clusters at multiple granularity levels. A dropdown selects
the level (L3 = coarse ~50 regions, L2 = medium ~800, L1 = fine ~14K). Each region bubble shows:
- **Size** = number of posts  
- **Color** = depression ratio (red = high depression, blue = control-dominated)
- **Hover** = representative posts + top c-TF-IDF words + symptom distribution within the region

This is the drill-down: click-equivalent via rich hover that shows the *clinical content* of each cluster.

In [10]:
# Build region data for multiple levels using region_assignments()
def build_region_data(level, max_regions=50, n_text_samples=50):
    """Build region metadata using a single-pass assignment scan."""
    t0 = time.perf_counter()
    
    regions = index.regions(level=level)
    regions_sorted = sorted(regions, key=lambda r: r[2], reverse=True)[:max_regions]
    print(f'  Level {level}: {len(regions)} total, using top {len(regions_sorted)}')
    
    print(f'  Running region_assignments...')
    all_assignments = index.region_assignments(level=level)
    print(f'  Assignments done in {time.perf_counter()-t0:.1f}s')
    
    # Vectorized centroid → centered anchor projection
    centroids = np.array([c for _, c, _ in regions_sorted], dtype=np.float32)
    centroids_c = centroids - global_centroid[np.newaxis, :]
    c_norms = np.linalg.norm(centroids_c, axis=1, keepdims=True) + 1e-8
    centroids_c = centroids_c / c_norms
    cos_sim = centroids_c @ anchor_matrix_centered.T
    all_dists = 1.0 - cos_sim
    top_symptom_indices = np.argmin(all_dists[:, :9], axis=1)
    
    region_rows = []
    for idx, (rid, centroid, n_members) in enumerate(regions_sorted):
        members = all_assignments.get(rid, [])
        
        labels = [user_label.get(id_to_user.get(eid), 'unknown') for eid, ts in members[:100]]
        dep_count = sum(1 for l in labels if l == 'depression')
        dep_ratio = dep_count / max(len(labels), 1)
        
        # Try more members to get enough text hits
        sample_texts = []
        for eid, ts in members[:n_text_samples]:
            uid = id_to_user.get(eid, '')
            if uid:
                txt = get_text(uid, ts, max_len=100)
                if txt and len(txt) > 10:
                    sample_texts.append(txt)
                    if len(sample_texts) >= 5:
                        break
        
        region_rows.append({
            'region_id': rid,
            'n_members': n_members,
            'centroid': centroid,
            'dep_ratio': dep_ratio,
            'top_symptom': SYMPTOM_KEYS[top_symptom_indices[idx]],
            'symptom_dists': all_dists[idx].tolist(),
            'sample_texts': sample_texts,
        })
    
    with_text = sum(1 for r in region_rows if r['sample_texts'])
    print(f'  Done: {len(region_rows)} regions, {with_text} with text ({time.perf_counter()-t0:.1f}s)')
    return region_rows

print('Building region data for levels 1, 2, 3...')
regions_l3_data = build_region_data(level=3, max_regions=100)
regions_l2_data = build_region_data(level=2, max_regions=150)
regions_l1_data = build_region_data(level=1, max_regions=200)

# c-TF-IDF labels
def compute_tfidf_labels(region_data, n_words=5):
    corpus = {r['region_id']: ' '.join(r['sample_texts'])
              for r in region_data if len(' '.join(r['sample_texts'])) > 30}
    if not corpus:
        for r in region_data:
            r['tfidf_label'] = '(no text)'
        return
    tfidf = TfidfVectorizer(max_features=3000, stop_words='english', min_df=1, max_df=0.9)
    tfidf_matrix = tfidf.fit_transform(corpus.values())
    names = tfidf.get_feature_names_out()
    labels = {}
    for i, rid in enumerate(corpus.keys()):
        top_idx = tfidf_matrix[i].toarray().flatten().argsort()[-n_words:][::-1]
        labels[rid] = ', '.join([names[j] for j in top_idx])
    for r in region_data:
        r['tfidf_label'] = labels.get(r['region_id'], '(no text)')

for rd in [regions_l3_data, regions_l2_data, regions_l1_data]:
    compute_tfidf_labels(rd)
print('c-TF-IDF labels computed for all levels')

Building region data for levels 1, 2, 3...
  Level 3: 356 total, using top 100
  Running region_assignments...
  Assignments done in 28.1s
  Done: 100 regions, 100 with text (28.1s)
  Level 2: 5249 total, using top 150
  Running region_assignments...
  Assignments done in 44.5s
  Done: 150 regions, 150 with text (44.5s)
  Level 1: 84788 total, using top 200
  Running region_assignments...
  Assignments done in 74.2s
  Done: 200 regions, 200 with text (74.2s)
c-TF-IDF labels computed for all levels


In [11]:
# Interactive HNSW Explorer — 3 levels with PCA layout
from sklearn.decomposition import PCA

def make_region_trace(region_data, pca_model=None, visible=True):
    """Region bubbles in PCA space with clinical hover content."""
    centroids = np.array([r['centroid'] for r in region_data], dtype=np.float32)
    
    if pca_model is None:
        pca_model = PCA(n_components=2, random_state=42)
        coords = pca_model.fit_transform(centroids)
    else:
        coords = pca_model.transform(centroids)
    
    sizes = [np.sqrt(r['n_members']) * 2 for r in region_data]
    colors = [r['dep_ratio'] for r in region_data]
    
    hover_texts = []
    for r in region_data:
        posts_preview = '<br>'.join([f'• {t[:80]}' for t in r['sample_texts'][:3]])
        if not posts_preview:
            posts_preview = '(no text available)'
        symptom_bar = ' | '.join([
            f'{SYMPTOM_KEYS[j]}: {1-r["symptom_dists"][j]:.3f}'
            for j in np.argsort(r['symptom_dists'][:9])[:3]
        ])
        keywords = r.get('tfidf_label', '(no keywords)')
        hover_texts.append(
            f"<b>Region {r['region_id']}</b> — {r['n_members']:,} posts<br>"
            f"Depression ratio: {r['dep_ratio']:.1%}<br>"
            f"Nearest symptom: <b>{r['top_symptom']}</b><br>"
            f"Top anchors: {symptom_bar}<br>"
            f"Keywords: {keywords}<br>"
            f"<br><b>Sample posts:</b><br>{posts_preview}"
        )
    
    return go.Scatter(
        x=coords[:, 0], y=coords[:, 1],
        mode='markers',
        marker=dict(
            size=sizes, color=colors,
            colorscale='RdBu_r', cmin=0, cmax=1,
            colorbar=dict(title='Depression<br>Ratio'),
            line=dict(width=1, color='white'),
            opacity=0.8,
        ),
        text=hover_texts,
        hovertemplate='%{text}<extra></extra>',
        visible=visible,
    ), pca_model

fig = go.Figure()

trace_l3, _ = make_region_trace(regions_l3_data, visible=True)
trace_l2, _ = make_region_trace(regions_l2_data, visible=False)
trace_l1, _ = make_region_trace(regions_l1_data, visible=False)
fig.add_trace(trace_l3)
fig.add_trace(trace_l2)
fig.add_trace(trace_l1)

fig.update_layout(
    title='HNSW Semantic Regions — Hover for Clinical Content',
    xaxis_title='PCA 1 (semantic spread)',
    yaxis_title='PCA 2 (semantic spread)',
    updatemenus=[dict(
        type='dropdown', direction='down',
        x=0.02, y=0.98, xanchor='left', yanchor='top',
        buttons=[
            dict(label=f'Level 3 — Coarse ({len(regions_l3_data)} regions)',
                 method='update', args=[{'visible': [True, False, False]}]),
            dict(label=f'Level 2 — Medium ({len(regions_l2_data)} regions)',
                 method='update', args=[{'visible': [False, True, False]}]),
            dict(label=f'Level 1 — Fine ({len(regions_l1_data)} regions)',
                 method='update', args=[{'visible': [False, False, True]}]),
        ],
        bgcolor='rgba(50,50,50,0.8)',
        font=dict(color='white'),
    )],
    width=1000, height=700,
    template='plotly_dark',
    annotations=[dict(
        text='Size = members | Color = depression ratio | Hover = symptom profile + keywords + posts',
        x=0.5, y=-0.08, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='gray'),
    )],
)
fig.show()

---
## 5. Clinical Timeline with Text Attribution

**Clinical question:** When did a user's language change, and what changed?

This is the key panel that bridges temporal analytics with clinical content. For a selected user:
1. **Velocity in anchor space** — how fast are they moving between symptoms
2. **Changepoints** — detected regime transitions
3. **Text attribution** — actual posts before and after each changepoint, showing *what* changed in language
4. **Symptom heatmap** — time × symptom matrix showing which symptoms intensify when

In [12]:
def clinical_timeline(uid, window_posts=5):
    """Clinical timeline with centered anchor projection + text attribution."""
    eid = user_to_id[uid]
    traj = get_traj(uid)
    if len(traj) < 20:
        print(f'{uid}: insufficient data ({len(traj)} posts)')
        return None
    
    label = user_label[uid]
    posts_df = user_posts_with_text(uid, traj)
    
    projected = project_centered(traj)
    timestamps = np.array([ts for ts, _ in traj])
    distances = np.array([dists for _, dists in projected])
    proximities = 1.0 - distances
    times_dt = [pd.Timestamp(ts, unit='s') for ts in timestamps]
    
    window = max(3, min(15, len(timestamps) // 5))
    prox_smooth = pd.DataFrame(proximities, columns=SYMPTOM_LABELS).rolling(window, center=True).mean().bfill().ffill().values
    
    vel_data = []
    for i in range(1, len(traj)):
        _, cos_drift, _ = cvx.drift(traj[i-1][1], traj[i][1], top_n=1)
        dt = max(1, traj[i][0] - traj[i-1][0])
        vel_data.append({'time': times_dt[i], 'velocity': cos_drift / dt, 'drift': cos_drift})
    df_vel = pd.DataFrame(vel_data)
    
    symptom_traj = [(int(timestamps[i]), [float(x) for x in distances[i][:9]]) for i in range(len(traj))]
    try:
        changepoints = cvx.detect_changepoints(
            eid, symptom_traj,
            penalty=3.0 * np.log(len(symptom_traj)),
            min_segment_len=max(5, len(symptom_traj) // 20),
        )
    except:
        changepoints = []
    
    cp_attributions = []
    for cp_ts, severity in changepoints:
        cp_idx = np.searchsorted(timestamps, cp_ts)
        before_start = max(0, cp_idx - window_posts)
        after_end = min(len(traj), cp_idx + window_posts)
        
        before_texts = [get_text(uid, traj[i][0], 150) for i in range(before_start, cp_idx)]
        after_texts = [get_text(uid, traj[i][0], 150) for i in range(cp_idx, after_end)]
        
        before_prox = proximities[before_start:cp_idx].mean(axis=0) if cp_idx > before_start else proximities[0]
        after_prox = proximities[cp_idx:after_end].mean(axis=0) if after_end > cp_idx else proximities[-1]
        shift = after_prox - before_prox
        top_increasing = np.argsort(shift)[-3:][::-1]
        top_decreasing = np.argsort(shift)[:3]
        
        cp_attributions.append({
            'timestamp': pd.Timestamp(cp_ts, unit='s'),
            'severity': severity,
            'before_texts': [t for t in before_texts if t],
            'after_texts': [t for t in after_texts if t],
            'shift': shift,
            'top_increasing': [(SYMPTOM_LABELS[i], shift[i]) for i in top_increasing if shift[i] > 0.001],
            'top_decreasing': [(SYMPTOM_LABELS[i], shift[i]) for i in top_decreasing if shift[i] < -0.001],
        })
    
    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True,
        subplot_titles=[
            'Symptom Proximity Heatmap (centered, time × DSM-5)',
            'Velocity in Embedding Space',
            'Changepoint Attribution',
            'Post Content (hover for text)',
        ],
        row_heights=[0.35, 0.2, 0.25, 0.2], vertical_spacing=0.06,
    )
    
    fig.add_trace(go.Heatmap(
        z=prox_smooth.T, x=times_dt, y=SYMPTOM_LABELS, colorscale='RdYlBu_r',
        hovertemplate='%{x}<br>%{y}: %{z:.3f}<extra></extra>',
        colorbar=dict(title='Proximity', x=1.02, len=0.3, y=0.85),
    ), row=1, col=1)
    
    for cp in cp_attributions:
        fig.add_vline(x=cp['timestamp'], line_dash='dot', line_color=C_ACCENT, line_width=2, row=1, col=1)
    
    if len(df_vel) > 0:
        fig.add_trace(go.Scatter(
            x=df_vel['time'], y=df_vel['velocity'], mode='lines',
            line=dict(color=C_DEP, width=1), showlegend=False,
        ), row=2, col=1)
        if len(df_vel) > 10:
            smooth_vel = df_vel['velocity'].rolling(min(20, len(df_vel)//3), center=True).mean()
            fig.add_trace(go.Scatter(
                x=df_vel['time'], y=smooth_vel, mode='lines',
                line=dict(color='yellow', width=2), showlegend=False,
            ), row=2, col=1)
    
    for cp in cp_attributions:
        fig.add_vline(x=cp['timestamp'], line_dash='dot', line_color=C_ACCENT, line_width=2, row=2, col=1)
    
    for i, cp in enumerate(cp_attributions):
        before_preview = '<br>'.join([f'• {t[:100]}' for t in cp['before_texts'][:3]])
        after_preview = '<br>'.join([f'• {t[:100]}' for t in cp['after_texts'][:3]])
        increasing_str = ', '.join([f'{name} ↑{val:+.3f}' for name, val in cp['top_increasing']])
        decreasing_str = ', '.join([f'{name} ↓{val:+.3f}' for name, val in cp['top_decreasing']])
        hover = (
            f'<b>Changepoint {i+1}</b> (severity: {cp["severity"]:.3f})<br><br>'
            f'<b>Symptoms increasing:</b> {increasing_str or "none"}<br>'
            f'<b>Symptoms decreasing:</b> {decreasing_str or "none"}<br><br>'
            f'<b>BEFORE:</b><br>{before_preview}<br><br>'
            f'<b>AFTER:</b><br>{after_preview}'
        )
        fig.add_trace(go.Scatter(
            x=[cp['timestamp']], y=[cp['severity']], mode='markers+text',
            marker=dict(size=15, color=C_ACCENT, symbol='diamond', line=dict(width=2, color='white')),
            text=[f'CP{i+1}'], textposition='top center',
            textfont=dict(color=C_ACCENT, size=10),
            hovertext=[hover], hovertemplate='%{hovertext}<extra></extra>', showlegend=False,
        ), row=3, col=1)
    
    post_hours = [pd.Timestamp(ts, unit='s').hour for ts, _ in traj]
    fig.add_trace(go.Scatter(
        x=times_dt, y=[1] * len(traj), mode='markers',
        marker=dict(size=5, color=post_hours, colorscale='Twilight',
                   colorbar=dict(title='Hour', x=1.08, len=0.15, y=0.1)),
        text=posts_df['text_short'].tolist(),
        hovertemplate='<b>%{text}</b><br>%{x}<extra></extra>', showlegend=False,
    ), row=4, col=1)
    
    for cp in cp_attributions:
        fig.add_vline(x=cp['timestamp'], line_dash='dot', line_color=C_ACCENT, line_width=2, row=4, col=1)
    
    hurst = cvx.hurst_exponent(traj) if len(traj) >= 20 else 0.5
    fig.update_layout(
        title=f'Clinical Timeline — {uid} ({label}) | {len(traj)} posts | H={hurst:.2f} | {len(changepoints)} changepoints',
        height=1100, width=1050, template='plotly_dark', showlegend=False,
    )
    fig.update_yaxes(title_text='Symptom', row=1, col=1)
    fig.update_yaxes(title_text='|dv/dt|', row=2, col=1)
    fig.update_yaxes(title_text='Severity', row=3, col=1)
    fig.update_yaxes(visible=False, row=4, col=1)
    fig.show()
    
    if cp_attributions:
        print(f'\n{"="*80}')
        print(f'TEXT ATTRIBUTION SUMMARY — {uid} ({label})')
        print(f'{"="*80}')
        for i, cp in enumerate(cp_attributions):
            print(f'\n--- Changepoint {i+1} at {cp["timestamp"]} (severity: {cp["severity"]:.3f}) ---')
            if cp['top_increasing']:
                print(f'  Symptoms INCREASING: {", ".join([f"{n} ({v:+.3f})" for n, v in cp["top_increasing"]])}')
            if cp['top_decreasing']:
                print(f'  Symptoms DECREASING: {", ".join([f"{n} ({v:+.3f})" for n, v in cp["top_decreasing"]])}')
            print(f'  BEFORE:')
            for t in cp['before_texts'][:3]:
                print(f'    "{t[:120]}"')
            print(f'  AFTER:')
            for t in cp['after_texts'][:3]:
                print(f'    "{t[:120]}"')
    
    return cp_attributions

In [13]:
# Run clinical timeline for a depression user
case_dep = sample_dep[0]
cp_dep = clinical_timeline(case_dep)

In [14]:
# Compare with a control user
case_ctl = sample_ctl[0]
cp_ctl = clinical_timeline(case_ctl)

---
## 6. Semantic Polarization

**Clinical question:** Is this user's world shrinking?

Healthy users write about diverse topics (work, hobbies, relationships) — their embedding space 
is **dispersed**. Depressed users progressively narrow their focus (rumination, isolation) — 
their embedding space **contracts**. We track this contraction over time as a clinical signal.

- **Dispersion** = mean pairwise cosine distance within a sliding window of posts
- **Contraction** = dispersion at window N / dispersion at window 0 (ratio < 1 = shrinking)

In [15]:
def compute_dispersion_timeline(uid, window_size=30, step=5):
    """Compute semantic dispersion over sliding windows."""
    traj = get_traj(uid)
    if len(traj) < window_size:
        return None
    
    results = []
    for start in range(0, len(traj) - window_size + 1, step):
        window_vecs = np.array([vec for _, vec in traj[start:start + window_size]])
        norms = np.linalg.norm(window_vecs, axis=1, keepdims=True) + 1e-8
        normed = window_vecs / norms
        sim_matrix = normed @ normed.T
        triu = np.triu_indices(len(normed), k=1)
        dispersion = 1.0 - float(np.mean(sim_matrix[triu]))
        
        mid_ts = traj[start + window_size // 2][0]
        try:
            t = pd.Timestamp(mid_ts, unit='s')
        except (ValueError, OverflowError):
            continue
        results.append({'time': t, 'dispersion': dispersion, 'window_start': start})
    
    return pd.DataFrame(results) if results else None

# Compute for sample users
print('Computing polarization timelines...')
fig = go.Figure()

for uid in list(sample_dep[:5]) + list(sample_ctl[:5]):
    disp = compute_dispersion_timeline(uid, window_size=30, step=3)
    if disp is None or len(disp) < 3:
        continue
    label = user_label[uid]
    color = C_DEP if label == 'depression' else C_CTL
    disp['ratio'] = disp['dispersion'] / (disp['dispersion'].iloc[0] + 1e-8)
    
    fig.add_trace(go.Scatter(
        x=disp['time'], y=disp['ratio'], mode='lines',
        line=dict(color=color, width=1.5), opacity=0.7,
        name=f'{uid} ({label[0].upper()})',
        hovertemplate=f'{uid}<br>Dispersion ratio: ' + '%{y:.3f}<br>%{x}<extra></extra>',
    ))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray', opacity=0.5,
              annotation_text='No change (ratio=1)', annotation_position='bottom right')
fig.update_layout(
    title='Semantic Polarization — Dispersion Ratio Over Time (< 1 = world shrinking)',
    xaxis_title='Time', yaxis_title='Dispersion Ratio (relative to first window)',
    width=1000, height=500, template='plotly_dark',
)
fig.show()

Computing polarization timelines...


In [16]:
# Population-level polarization: box plot of final dispersion ratio
print('Computing population dispersion ratios...')
pop_ratios = []
sample_uids = list(user_post_counts.keys())[:500]
for uid in sample_uids:
    disp = compute_dispersion_timeline(uid, window_size=30, step=10)
    if disp is None or len(disp) < 3:
        continue
    ratio = disp['dispersion'].iloc[-1] / (disp['dispersion'].iloc[0] + 1e-8)
    pop_ratios.append({
        'user_id': uid, 'label': user_label[uid],
        'final_ratio': ratio, 'n_posts': user_post_counts[uid],
    })

df_polar = pd.DataFrame(pop_ratios)
print(f'Computed for {len(df_polar)} users')

fig = go.Figure()
for label, color in [('depression', C_DEP), ('control', C_CTL)]:
    vals = df_polar[df_polar['label'] == label]['final_ratio']
    fig.add_trace(go.Box(y=vals, name=label.capitalize(), marker_color=color, boxmean='sd'))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray')
fig.update_layout(
    title='Semantic Polarization Distribution — Final Dispersion Ratio',
    yaxis_title='Dispersion Ratio (last/first window)',
    width=600, height=500, template='plotly_dark',
    annotations=[dict(
        text='< 1 = semantic contraction (fewer topics)',
        x=0.5, y=-0.12, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='gray'),
    )],
)
fig.show()

print(f'\nMean ratio — Depression: {df_polar[df_polar["label"]=="depression"]["final_ratio"].mean():.3f}, '
      f'Control: {df_polar[df_polar["label"]=="control"]["final_ratio"].mean():.3f}')

Computing population dispersion ratios...
Computed for 439 users



Mean ratio — Depression: 0.996, Control: 1.021


---
## 7. Cohort Divergence — Top Discriminative Symptoms

**Clinical question:** Which symptoms diverge most between groups, and when?

Reuses the per-symptom temporal data from Section 3 to show the **difference** (dep − ctl)
over time for the most discriminative symptoms.

In [17]:
# Divergence heatmap + top-4 line plot — reuse data from section 3
divergence = dep_curves - ctl_curves  # (n_time_bins, 10) — positive = dep closer

fig = go.Figure(data=go.Heatmap(
    z=divergence.T,
    x=[f'{int(t)}%' for t in time_pct], y=SYMPTOM_LABELS,
    colorscale='RdBu_r', zmid=0,
    hovertemplate='Time: %{x}<br>%{y}<br>Divergence: %{z:.4f}<extra></extra>',
    colorbar=dict(title='Dep−Ctl<br>Proximity'),
))
fig.update_layout(
    title='Cohort Divergence in Symptom Space — Depression vs Control Over Normalized Time',
    xaxis_title='% of User Post History', yaxis_title='DSM-5 Symptom',
    width=1000, height=500, template='plotly_dark',
    annotations=[dict(
        text='Red = depression cohort CLOSER to symptom than control | Blue = control closer',
        x=0.5, y=-0.15, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='gray'),
    )],
)
fig.show()

In [18]:
# Line plot: per-symptom divergence over time (top 4 most discriminative)
top_symptoms = np.argsort(np.abs(divergence).mean(axis=0))[-4:][::-1]

fig = go.Figure()
colors = ['#e74c3c', '#f39c12', '#9b59b6', '#2ecc71']

for idx, sym_idx in enumerate(top_symptoms):
    fig.add_trace(go.Scatter(
        x=time_pct, y=divergence[:, sym_idx],
        mode='lines+markers',
        name=SYMPTOM_LABELS[sym_idx],
        line=dict(color=colors[idx], width=2.5),
        marker=dict(size=5),
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(
    title='Top 4 Discriminative Symptoms — Divergence Over Normalized Time',
    xaxis_title='% of User Post History',
    yaxis_title='Divergence (Dep − Ctl proximity)',
    width=900, height=450,
    template='plotly_dark',
    annotations=[dict(
        text='> 0 = depression users are closer to this symptom than controls',
        x=0.5, y=-0.15, xref='paper', yref='paper',
        showarrow=False, font=dict(size=10, color='gray'),
    )],
)
fig.show()

---
## 8. User Clinical Profile

**Clinical question:** What is the complete clinical picture of this user?

This is the unified view that connects all panels. Call `clinical_profile(user_id)` for any user
to get:
- Symptom radar (where they are)
- Symptom drift radar (where they're heading)  
- Clinical timeline (when things changed + what changed)
- Polarization curve (is their world shrinking?)
- Summary card with clinical metrics

In [19]:
def clinical_profile(uid):
    """Complete clinical profile for a single user."""
    traj = get_traj(uid)
    if len(traj) < 20:
        print(f'{uid}: insufficient data')
        return
    
    label = user_label[uid]
    summary = user_summaries.get(uid)
    if summary is None:
        print(f'{uid}: no anchor summary')
        return
    
    hurst = cvx.hurst_exponent(traj) if len(traj) >= 20 else 0.5
    ts_list = [t for t, _ in traj]
    ef = cvx.event_features(ts_list)
    
    disp_df = compute_dispersion_timeline(uid, window_size=30, step=5)
    polarization_ratio = (disp_df['dispersion'].iloc[-1] / (disp_df['dispersion'].iloc[0] + 1e-8)) if disp_df is not None and len(disp_df) > 2 else 1.0
    
    trends = summary['trend']
    top_approaching_idx = np.argmin(trends[:9])
    top_approaching = SYMPTOM_KEYS[top_approaching_idx]
    approach_rate = -trends[top_approaching_idx]
    
    means = summary['mean']
    closest_idx = np.argmin(means[:9])
    closest_symptom = SYMPTOM_KEYS[closest_idx]
    
    print(f'{"="*80}')
    print(f'CLINICAL PROFILE — {uid} ({label.upper()})')
    print(f'{"="*80}')
    print(f'  Posts: {len(traj)}')
    print(f'  Hurst exponent: {hurst:.3f} ({"persistent" if hurst > 0.5 else "anti-persistent"})')
    print(f'  Burstiness: {ef["burstiness"]:.3f}')
    print(f'  Circadian strength: {ef["circadian_strength"]:.3f}')
    print(f'  Polarization ratio: {polarization_ratio:.3f} ({"contracting" if polarization_ratio < 0.95 else "stable" if polarization_ratio < 1.05 else "expanding"})')
    print(f'  Closest symptom: {closest_symptom} (distance: {means[closest_idx]:.3f})')
    print(f'  Fastest approach: {top_approaching} (rate: {approach_rate:.5f}/post)')
    print()
    
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'polar'}, {'type': 'polar'}]],
        subplot_titles=['Current Symptom Profile', 'Drift Direction (approaching)'],
    )
    
    proximity = [1.0 - d for d in means]
    prox_closed = proximity + [proximity[0]]
    fig.add_trace(go.Scatterpolar(
        r=prox_closed, theta=radar_labels, fill='toself', name='User',
        line=dict(color=C_DEP if label == 'depression' else C_CTL, width=3),
        fillcolor=f'rgba({"231,76,60" if label == "depression" else "52,152,219"}, 0.2)',
    ), row=1, col=1)
    
    pop_mean = dep_mean if label == 'depression' else ctl_mean
    pop_closed = list(pop_mean) + [pop_mean[0]]
    fig.add_trace(go.Scatterpolar(
        r=pop_closed, theta=radar_labels,
        name=f'{label.capitalize()} avg',
        line=dict(color='gray', width=1, dash='dot'),
    ), row=1, col=1)
    
    drift_toward = [-t for t in trends]
    drift_closed = list(drift_toward) + [drift_toward[0]]
    fig.add_trace(go.Scatterpolar(
        r=drift_closed, theta=radar_labels, fill='toself', name='Drift',
        line=dict(color=C_WARN, width=3),
        fillcolor='rgba(243, 156, 18, 0.2)',
    ), row=1, col=2)
    
    fig.update_layout(
        title=f'{uid} — Symptom Profile & Drift',
        width=1000, height=500, template='plotly_dark',
    )
    fig.show()
    
    if disp_df is not None and len(disp_df) > 2:
        disp_df['ratio'] = disp_df['dispersion'] / (disp_df['dispersion'].iloc[0] + 1e-8)
        fig_pol = go.Figure()
        fig_pol.add_trace(go.Scatter(
            x=disp_df['time'], y=disp_df['ratio'], mode='lines',
            line=dict(color=C_DEP if label == 'depression' else C_CTL, width=2.5),
            fill='tonexty' if polarization_ratio < 0.95 else None,
        ))
        fig_pol.add_hline(y=1.0, line_dash='dash', line_color='gray')
        status = 'CONTRACTING' if polarization_ratio < 0.95 else 'STABLE' if polarization_ratio < 1.05 else 'EXPANDING'
        fig_pol.update_layout(
            title=f'Semantic Polarization — {uid} | Status: {status} (ratio={polarization_ratio:.3f})',
            xaxis_title='Time', yaxis_title='Dispersion Ratio',
            width=900, height=350, template='plotly_dark',
        )
        fig_pol.show()
    
    print()
    clinical_timeline(uid)

In [20]:
# === RUN CLINICAL PROFILE FOR A DEPRESSION USER ===
clinical_profile(sample_dep[0])

CLINICAL PROFILE — e2017train_train_subject6146 (DEPRESSION)
  Posts: 337
  Hurst exponent: 0.645 (persistent)
  Burstiness: 0.636
  Circadian strength: 0.290
  Polarization ratio: 1.349 (expanding)
  Closest symptom: fatigue (distance: 0.765)
  Fastest approach: psychomotor (rate: -0.00011/post)



In [21]:
# === RUN CLINICAL PROFILE FOR A CONTROL USER (comparison) ===
clinical_profile(sample_ctl[0])

CLINICAL PROFILE — e2022_subject7827 (CONTROL)
  Posts: 242
  Hurst exponent: 0.701 (persistent)
  Burstiness: 0.611
  Circadian strength: 0.268
  Polarization ratio: 1.274 (expanding)
  Closest symptom: fatigue (distance: 0.808)
  Fastest approach: suicidal_ideation (rate: 0.00032/post)



In [22]:
# === EXPLORE ANY USER ===
# Change the user_id below to inspect any user in the dataset.
print('Depression users with 50-500 posts (sample):')
for uid in dep_users[:20]:
    print(f'  {uid}: {user_post_counts[uid]} posts')
print(f'\n  ... {len(dep_users)} total depression users available')
print(f'  ... {len(ctl_users)} total control users available')
print('\nCall: clinical_profile("user_id_here")')

Depression users with 50-500 posts (sample):
  e2022_subject1457: 142 posts
  e2022_subject2365: 125 posts
  e2022_subject2713: 129 posts
  e2022_subject2716: 53 posts
  e2022_subject2745: 153 posts
  e2022_subject2870: 60 posts
  e2022_subject3092: 235 posts
  e2022_subject3362: 135 posts
  e2022_subject3495: 302 posts
  e2022_subject354: 50 posts
  e2022_subject357: 93 posts
  e2022_subject4153: 88 posts
  e2022_subject4161: 108 posts
  e2022_subject4928: 91 posts
  e2022_subject5463: 189 posts
  e2022_subject5731: 109 posts
  e2022_subject5770: 60 posts
  e2022_subject6074: 99 posts
  e2022_subject6185: 134 posts
  e2022_subject6724: 65 posts

  ... 101 total depression users available
  ... 763 total control users available

Call: clinical_profile("user_id_here")


---
## Summary

### Paradigm shift: statistical → clinical

| B1 Explorer (statistical) | B3 Dashboard (clinical) |
|--------------------------|------------------------|
| PCA axes (uninterpretable) | DSM-5 symptom axes (anhedonia, worthlessness, ...) |
| Static HNSW bubbles | Interactive level selector + rich hover with posts & symptom distribution |
| Velocity magnitude (how fast?) | Velocity + text attribution (what changed in language?) |
| Hurst/drift statistics | Semantic polarization (is their world shrinking?) |
| Per-user scatter | Cohort divergence heatmap (when do groups separate, in which symptoms?) |
| 5-panel deep-dive | Full clinical profile card with radar, polarization, timeline |

### CVX functions used

| Function | Panel | What it reveals |
|----------|-------|----------------|
| `project_to_anchors` | Radar, Trajectory, Cohort | Distance to each DSM-5 symptom |
| `anchor_summary` | Radar, Profile | Mean/min/trend per symptom |
| `velocity`, `drift` | Timeline | Rate and direction of language change |
| `detect_changepoints` | Timeline | Regime transitions with text attribution |
| `hurst_exponent` | Profile | Long-range dependence of trajectory |
| `event_features` | Profile | Posting pattern (burstiness, circadian) |
| `regions`, `region_assignments` | HNSW Explorer | Semantic clusters with clinical content |

### Key insight

**The same CVX temporal analytics that work on raw embeddings also work in anchor-projected space.**
`project_to_anchors()` doesn't just reduce dimensions — it transforms the coordinate system so that
every measurement (velocity, drift, changepoints) has direct clinical interpretation.

A velocity spike in PCA space means "something changed." A velocity spike in anchor space means
"the user suddenly shifted from sleep complaints to suicidal ideation."